In [ ]:
# Step 1: match_sop(Diagnose the incident for the Online Boutique system. Incident window: 2024-01-19 23:08:24 to 2024-01-19 23:38:24 UTC. A
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.55)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.50)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Level and Root Selection SOP (score 0.41)
  Name: RCA-Agent Level and Root Selection SOP
Steps:
1. component_analyze(start_time, end_time): identify the strongest faulty candidates and their telemetry sources.
2. collect_trace(start_time, end_time): separate faulty upstream/downstream candidates when trace telemetry is available.
3. get_logs(None, start_time, end_time): check whether logs support the same component and reason as the metric/trace evidence.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 2: generate_sop_code for 'RCA-Agent Metric Fault Identification SOP'
start_time = '2024-01-19 23:08:24'
end_time = '2024-01-19 23:38:24'
metrics_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
component_comparison = component_analyze(start_time, end_time)
answer = metrics_evidence + '\n\n' + component_comparison

```
Generated code for SOP 'RCA-Agent Metric Fault Identification SOP':
start_time = '2024-01-19 23:08:24'
end_time = '2024-01-19 23:38:24'
metrics_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
component_comparison = component_analyze(start_time, end_time)
answer = metrics_evidence + '\n\n' + component_comparison
```

In [ ]:
# Step 3: run_sop
start_time = '2024-01-19 23:08:24'
end_time = '2024-01-19 23:38:24'
metrics_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
component_comparison = component_analyze(start_time, end_time)
answer = metrics_evidence + '\n\n' + component_comparison

```
[ANOMALY] checkoutservice latency-50: rise to 1.016 at 2024-01-19 23:26:00 (median 0.04487, thr 0.8208, sev 28.8, n=2)
[ANOMALY] currencyservice latency-50: rise to 0.1022 at 2024-01-19 23:25:00 (median 0.003296, thr 0.102, sev 20.8, n=2)
[ANOMALY] emailservice mem: rise to 4.4e+07 at 2024-01-19 23:25:00 (median 4.397e+07, thr 4.398e+07, sev 10.3, n=1)
[ANOMALY] emailservice latency-90: rise to 0.004715 at 2024-01-19 23:20:00 (median 0.004609, thr 0.0047, sev 8.4, n=2)
[ANOMALY] emailservice latency-50: rise to 0.003064 at 2024-01-19 23:20:00 (median 0.003005, thr 0.003056, sev 8.4, n=2)
[ANOMALY] checkoutservice latency-90: rise to 2.187 at 2024-01-19 23:26:00 (median 0.2479, thr 2.024, sev 7.5, n=2)
[ANOMALY] currencyservice cpu: drop to 8.332 at 2024-01-19 23:22:00 (median 10.32, thr 9.346, sev 6.1, n=1)
[ANOMALY] recommendationservice mem: drop to 4.598e+07 at 2024-01-19 23:20:00 (median 4.604e+07, thr 4.6e+07, sev 5.8, n=4)
[ANOMALY] recommendationservice latency-50: drop to 0.007404 at 2024-01-19 23:17:00 (median 0.007468, thr 0.007421, sev 4.7, n=4)
[ANOMALY] productcatalogservice mem: drop to 1.034e+07 at 2024-01-19 23:20:00 (median 1.06e+07, thr 1.037e+07, sev 3.5, n=2)
[ANOMALY] currencyservice latency-90: rise to 0.2251 at 2024-01-19 23:26:00 (median 0.0441, thr 0.2238, sev 3.1, n=2)

Component anomaly overview (ranked):
recommendationservice  metric_anoms=2 (top: mem sev=5.8)  trace_anoms=10  log_anoms=1
checkoutservice  metric_anoms=2 (top: latency-50 sev=28.8)  trace_anoms=7  log_anoms=0
currencyservice  metric_anoms=3 (top: latency-50 sev=20.8)  trace_anoms=6  log_anoms=0
productcatalogservice  metric_anoms=1 (top: mem sev=3.5)  trace_anoms=6  log_anoms=0
emailservice  metric_anoms=3 (top: mem sev=10.3)  trace_anoms=3  log_anoms=0
```

In [ ]:
# Step 4: match_observation([ANOMALY] checkoutservice latency-50: rise to 1.016 at 2024-01-19 23:26:00 (median 0.04487, thr 0.8208, sev 28.8, n=2)
[
...[truncated])

```
ObAgent: The fault class is likely network latency.

Analysis: Multiple services (checkout, currency, email) show simultaneous, significant rises in service-level latencies while CPU and memory do not show correlated increases (CPU actually dropped in currencyservice; mem changes are minor). There are no disk or retransmit/error signals that would indicate packet loss or socket errors. To confirm, check network RTT/host NIC metrics and TCP retransmits/errors (whether_is_abnormal_metric on network rtt and tcp_retransmits) and collect a distributed trace of the checkout → currency call path.
JudgeAgent: No - The run ends with a hypothesis (network latency) but no single component from the candidate list was selected as the location, nor was a concrete telemetry reason (e.g., specific network RTT or TCP retransmit anomaly) pinned as the root cause. Multiple services show latency anomalies (checkoutservice latency-50/90 spikes, currencyservice latency rise), but the history stops before choosing one component and providing the required specific evidence.
```

In [ ]:
# Step 5: match_sop(Find SOPs for network latency diagnosis (RTT, tcp_retransmits, NIC errors) and trace collection for checkout → currency 
...[truncated])

```
Matched SOPs:
- RCA-Agent Trace Localization SOP (score 0.42)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Metric Fault Identification SOP (score 0.41) (already executed)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Log-Based Reason Analysis SOP (score 0.38)
  Name: RCA-Agent Log-Based Reason Analysis SOP
Steps:
1. get_logs(None, start_time, end_time): summarize log anomalies and raw error messages for candidate components in the incident window.
2. get_relevant_metric('error'): list error, resource, and traffic-related metric fields that may explain the logs.
3. whether_is_abnormal_metric(start_time, end_time, '', None): compare log evidence against metric anomalies and onset times.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 6: generate_sop_code for 'RCA-Agent Trace Localization SOP'
start_time = '2024-01-19 23:08:24'
end_time = '2024-01-19 23:38:24'
trace_summary = collect_trace(start_time, end_time, service=None)
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_summary + '\n\n' + metric_evidence

```
Generated code for SOP 'RCA-Agent Trace Localization SOP':
start_time = '2024-01-19 23:08:24'
end_time = '2024-01-19 23:38:24'
trace_summary = collect_trace(start_time, end_time, service=None)
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_summary + '\n\n' + metric_evidence
```

In [ ]:
# Step 7: run_sop
start_time = '2024-01-19 23:08:24'
end_time = '2024-01-19 23:38:24'
trace_summary = collect_trace(start_time, end_time, service=None)
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_summary + '\n\n' + metric_evidence

```
[TRACE] currencyservice trace.from_root.row_count: drop to 4 at 2024-01-19 23:11:00 (median 12, sev 46.4)
[TRACE] emailservice trace.from_root.row_count: drop to 8 at 2024-01-19 23:11:00 (median 24, sev 46.4)
[TRACE] paymentservice trace.from_root.row_count: drop to 4 at 2024-01-19 23:11:00 (median 12, sev 46.4)
[TRACE] recommendationservice trace.from_root.row_count: drop to 8 at 2024-01-19 23:11:00 (median 24, sev 46.4)
[TRACE] root trace.to_currencyservice.row_count: drop to 4 at 2024-01-19 23:11:00 (median 12, sev 46.4)
[TRACE] root trace.to_emailservice.row_count: drop to 8 at 2024-01-19 23:11:00 (median 24, sev 46.4)
[TRACE] root trace.to_paymentservice.row_count: drop to 4 at 2024-01-19 23:11:00 (median 12, sev 46.4)
[TRACE] root trace.to_recommendationservice.row_count: drop to 8 at 2024-01-19 23:11:00 (median 24, sev 46.4)
[TRACE] currencyservice trace.from_root.duration_p95: rise to 0.0005446 at 2024-01-19 23:24:00 (median 7.025e-05, sev 46.4)
[TRACE] root trace.to_currencyservice.duration_p95: rise to 0.0005446 at 2024-01-19 23:24:00 (median 7.025e-05, sev 46.4)
[TRACE] productcatalogservice trace.from_root.row_count: drop to 9 at 2024-01-19 23:11:00 (median 24, sev 44.6)
[TRACE] root trace.to_productcatalogservice.row_count: drop to 9 at 2024-01-19 23:11:00 (median 24, sev 44.6)
[TRACE] checkoutservice trace.from_checkoutservice.duration_mean: rise to 0.00798 at 2024-01-19 23:27:00 (median 0.003584, sev 38.9)
[TRACE] checkoutservice trace.to_checkoutservice.duration_mean: rise to 0.00798 at 2024-01-19 23:27:00 (median 0.003584, sev 38.9)
[TRACE] currencyservice trace.from_frontendservice.duration_mean: rise to 0.0002158 at 2024-01-19 23:27:00 (median 0.0001204, sev 29.2)
Most-downstream faulty component(s): currencyservice, emailservice, paymentservice, productcatalogservice, root

[ANOMALY] checkoutservice latency-50: rise to 1.016 at 2024-01-19 23:26:00 (median 0.04487, thr 0.8208, sev 28.8, n=2)
[ANOMALY] currencyservice latency-50: rise to 0.1022 at 2024-01-19 23:25:00 (median 0.003296, thr 0.102, sev 20.8, n=2)
[ANOMALY] emailservice mem: rise to 4.4e+07 at 2024-01-19 23:25:00 (median 4.397e+07, thr 4.398e+07, sev 10.3, n=1)
[ANOMALY] emailservice latency-90: rise to 0.004715 at 2024-01-19 23:20:00 (median 0.004609, thr 0.0047, sev 8.4, n=2)
[ANOMALY] emailservice latency-50: rise to 0.003064 at 2024-01-19 23:20:00 (median 0.003005, thr 0.003056, sev 8.4, n=2)
[ANOMALY] checkoutservice latency-90: rise to 2.187 at 2024-01-19 23:26:00 (median 0.2479, thr 2.024, sev 7.5, n=2)
[ANOMALY] currencyservice cpu: drop to 8.332 at 2024-01-19 23:22:00 (median 10.32, thr 9.346, sev 6.1, n=1)
[ANOMALY] recommendationservice mem: drop to 4.598e+07 at 2024-01-19 23:20:00 (median 4.604e+07, thr 4.6e+07, sev 5.8, n=4)
[ANOMALY] recommendationservice latency-50: drop to 0.007404 at 2024-01-19 23:17:00 (median 0.007468, thr 0.007421, sev 4.7, n=4)
[ANOMALY] productcatalogservice mem: drop to 1.034e+07 at 2024-01-19 23:20:00 (median 1.06e+07, thr 1.037e+07, sev 3.5, n=2)
[ANOMALY] currencyservice latency-90: rise to 0.2251 at 2024-01-19 23:26:00 (median 0.0441, thr 0.2238, sev 3.1, n=2)
```

In [ ]:
# Step 8: match_observation([TRACE] currencyservice trace.from_root.row_count: drop to 4 at 2024-01-19 23:11:00 (median 12, sev 46.4)
[TRACE] emails
...[truncated])

```
ObAgent: uncertain

Analysis: simultaneous, large drop in trace row_count from root to many downstream services (23:11) plus later rises in downstream p95/duration could be caused by network problems (packet loss/latency) or by the root process failing to send requests (CPU/sockets). To disambiguate, check:
- whether_is_abnormal_metric for TCP retransmits/packet-loss and network rtt between root and services,
- whether_is_abnormal_metric for socket errors/connection resets on root and downstream,
- whether_is_abnormal_metric for root service CPU utilization and error/5xx rates,
- collect_trace on root->downstream calls to see error types.
JudgeAgent: No - The run ended with a hypothesis of network latency but did not pick a single component from the candidate list nor provide a concrete telemetry reason (e.g., specific abnormal network RTT, TCP retransmits, or socket error metric). Multiple services show anomalies, but the exact location and a single anomalous KPI/log/trace as root cause were not pinned.
```

In [ ]:
# Step 9: match_sop(Find SOPs for network-related diagnostics and trace localization: check RTT, tcp_retransmits, NIC errors, socket errors,
...[truncated])

```
Matched SOPs:
- RCA-Agent Trace Localization SOP (score 0.57) (already executed)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Metric Fault Identification SOP (score 0.46) (already executed)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Log-Based Reason Analysis SOP (score 0.41)
  Name: RCA-Agent Log-Based Reason Analysis SOP
Steps:
1. get_logs(None, start_time, end_time): summarize log anomalies and raw error messages for candidate components in the incident window.
2. get_relevant_metric('error'): list error, resource, and traffic-related metric fields that may explain the logs.
3. whether_is_abnormal_metric(start_time, end_time, '', None): compare log evidence against metric anomalies and onset times.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 10: generate_sop_code for 'RCA-Agent Log-Based Reason Analysis SOP'
start_time = '2024-01-19 23:08:24'
end_time = '2024-01-19 23:38:24'
logs = get_logs(None, start_time, end_time)
relevant_metrics = get_relevant_metric('error')
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = logs + '\n\n' + relevant_metrics + '\n\n' + metric_evidence

```
Generated code for SOP 'RCA-Agent Log-Based Reason Analysis SOP':
start_time = '2024-01-19 23:08:24'
end_time = '2024-01-19 23:38:24'
logs = get_logs(None, start_time, end_time)
relevant_metrics = get_relevant_metric('error')
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = logs + '\n\n' + relevant_metrics + '\n\n' + metric_evidence
```

In [ ]:
# Step 11: run_sop
start_time = '2024-01-19 23:08:24'
end_time = '2024-01-19 23:38:24'
logs = get_logs(None, start_time, end_time)
relevant_metrics = get_relevant_metric('error')
metric_evidence = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = logs + '\n\n' + relevant_metrics + '\n\n' + metric_evidence

```
[LOG] recommendationservice log.row_count: drop to 186 at 2024-01-19 23:35:00 (median 495)

Metric KPIs matching 'error' (top 3):
- error
- workload
- mem

[ANOMALY] checkoutservice latency-50: rise to 1.016 at 2024-01-19 23:26:00 (median 0.04487, thr 0.8208, sev 28.8, n=2)
[ANOMALY] currencyservice latency-50: rise to 0.1022 at 2024-01-19 23:25:00 (median 0.003296, thr 0.102, sev 20.8, n=2)
[ANOMALY] emailservice mem: rise to 4.4e+07 at 2024-01-19 23:25:00 (median 4.397e+07, thr 4.398e+07, sev 10.3, n=1)
[ANOMALY] emailservice latency-90: rise to 0.004715 at 2024-01-19 23:20:00 (median 0.004609, thr 0.0047, sev 8.4, n=2)
[ANOMALY] emailservice latency-50: rise to 0.003064 at 2024-01-19 23:20:00 (median 0.003005, thr 0.003056, sev 8.4, n=2)
[ANOMALY] checkoutservice latency-90: rise to 2.187 at 2024-01-19 23:26:00 (median 0.2479, thr 2.024, sev 7.5, n=2)
[ANOMALY] currencyservice cpu: drop to 8.332 at 2024-01-19 23:22:00 (median 10.32, thr 9.346, sev 6.1, n=1)
[ANOMALY] recommendationservice mem: drop to 4.598e+07 at 2024-01-19 23:20:00 (median 4.604e+07, thr 4.6e+07, sev 5.8, n=4)
[ANOMALY] recommendationservice latency-50: drop to 0.007404 at 2024-01-19 23:17:00 (median 0.007468, thr 0.007421, sev 4.7, n=4)
[ANOMALY] productcatalogservice mem: drop to 1.034e+07 at 2024-01-19 23:20:00 (median 1.06e+07, thr 1.037e+07, sev 3.5, n=2)
[ANOMALY] currencyservice latency-90: rise to 0.2251 at 2024-01-19 23:26:00 (median 0.0441, thr 0.2238, sev 3.1, n=2)
```

In [ ]:
# Step 12: match_observation([LOG] recommendationservice log.row_count: drop to 186 at 2024-01-19 23:35:00 (median 495)

Metric KPIs matching 'error'
...[truncated])

```
ObAgent: The fault class is likely to be network latency.

Analysis: multiple services (checkout, currency) show simultaneous and large increases in latency percentiles around 23:25–23:26, while CPU and memory do not show corresponding resource spikes (currency CPU actually drops; memory changes are minor or decreases). There are no signs of disk pressure or socket errors in the data. This pattern (widespread latency rises without resource overload) is most consistent with elevated network latency.
JudgeAgent: No - Not found: the agents ended with a hypothesis of network latency but did not pick exactly one component from the candidate list nor provide a concrete telemetry reason (e.g., specific abnormal network RTT, TCP retransmits, or NIC/socket error). Multiple services (checkoutservice, currencyservice, emailservice) show latency anomalies, so the required single location and specific telemetry evidence are not pinned.
```